# Sim Flip Chip Distance - Route B - Ignore Port-Sheet Build Probe

Audience: SGB developers checking whether the big `sim_flip_chip_distance`
geometry can build conformal CAD when the Palace lumped-port sheet layer is
temporarily ignored.

Outline:

1. Locate the saved probe XAO and sidecars.
2. Inspect the measured build timing and engine-gate status.
3. Optionally rebuild the probe stack and XAO.

The main SGB build path still fails fast when `port_sheet_regions` are live.
This probe copies the stack metadata and moves layer `202/1` into ignored
layers only to isolate large-geometry backend and mesh behavior.


In [ ]:
from pathlib import Path
import json

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").is_file():
    if ROOT.parent == ROOT:
        raise RuntimeError("Could not locate repository root")
    ROOT = ROOT.parent

RUN_FOLDER = ROOT / "tutorials" / "runs" / "no_inset" / "sim_flip_chip_distance" / "route_b_ignore_port_sheet_probe"
METADATA_DIR = RUN_FOLDER / "metadata" / "semantic_geometry"
XAO_PATH = RUN_FOLDER / "geometry" / "semantic_geometry_route_b.xao"
PHYSICAL_GROUPS_PATH = METADATA_DIR / "04_export_physical_groups.json"

print("run folder:", RUN_FOLDER)
print("XAO exists:", XAO_PATH.is_file(), XAO_PATH)
print("physical groups exists:", PHYSICAL_GROUPS_PATH.is_file(), PHYSICAL_GROUPS_PATH)


## Observed Output

Current saved output:

- XAO: `tutorials/runs/no_inset/sim_flip_chip_distance/route_b_ignore_port_sheet_probe/geometry/semantic_geometry_route_b.xao`
- physical groups: `metadata/semantic_geometry/04_export_physical_groups.json`
- build timing: `91.14s`
- XAO size: `53,554,352` bytes
- engine gates: `2d_inset`, `volume_adjacency`, and `gmsh_brep` passed


In [ ]:
timing = json.loads((METADATA_DIR / "00_timing.json").read_text())
print("total_seconds:", timing["total_seconds"])
print("counts:")
for key, value in sorted(timing["counts"].items()):
    print(f"  {key}: {value}")

for name in [
    "engine_gate_2d_inset_coverage.json",
    "engine_gate_volume_adjacency_conformality.json",
    "engine_gate_gmsh_brep_conformality.json",
]:
    payload = json.loads((METADATA_DIR / name).read_text())
    print(name, payload.get("status"))


## Optional Rebuild

Leave `RUN_BUILD = False` when you only want to inspect saved output. Set it
to `True` to regenerate the ignore-port-sheet probe from the source GDS and
stack JSON.


In [ ]:
RUN_BUILD = False

if not RUN_BUILD:
    print("Set RUN_BUILD = True to rebuild the probe XAO.")
else:
    import shutil
    import time
    from semantic_geometry_builder import SemanticGeometryBuilder, build_gds_stack_geometry_input

    asset_dir = ROOT / "tutorials" / "assets"
    source_stack = asset_dir / "sim_flip_chip_distance.stack.json"
    stack = json.loads(source_stack.read_text())
    metadata = dict(stack.get("metadata", {}))
    port_layers = list(metadata.pop("port_sheet_source_layers", []))
    ignored = list(metadata.get("ignored_layout_layers", []))
    for record in port_layers:
        ignored.append({
            "layer": record["layer"],
            "datatype": record.get("datatype", 0),
            "reason": "Probe: ignored Palace lumped-port sheet layer to test backend/mesh without port-sheet lowering.",
        })
    metadata["ignored_layout_layers"] = ignored
    stack["metadata"] = metadata

    if RUN_FOLDER.exists():
        shutil.rmtree(RUN_FOLDER)
    RUN_FOLDER.mkdir(parents=True)
    probe_stack = RUN_FOLDER / "sim_flip_chip_distance.ignore_port_sheet.stack.json"
    probe_stack.write_text(json.dumps(stack, indent=2, sort_keys=True) + "\n")

    started = time.perf_counter()
    build_input = build_gds_stack_geometry_input(
        gds_file=asset_dir / "sim_flip_chip_distance.gds",
        stack_file=probe_stack,
        top_cell_name="sim_flip_chip_distance_fixture",
        metadata={"inset_routes": []},
    )
    print("adapter_s:", round(time.perf_counter() - started, 3))
    print("polygons:", len(build_input.polygons))
    print("entities:", len(build_input.entities))
    print("port_regions:", len(build_input.port_sheet_regions))

    build_started = time.perf_counter()
    groups = SemanticGeometryBuilder().build(build_input, route="B", run_folder=RUN_FOLDER)
    print("build_s:", round(time.perf_counter() - build_started, 3))
    print("groups:", len(groups))
    print("xao:", XAO_PATH)
